In [0]:
#ler a camada silver
df_silver = spark.table(
    "mercado_financeiro.silver.ibvespa_acoes"
)

In [0]:
#extrair ticker unicos
dim_ticker = (
    df_silver
   .select("Codigo")
   .distinct()
)

In [0]:
#visualizando a tabela
dim_ticker.show()

In [0]:
dim_ticker.printSchema()

In [0]:
#salvar na camada gold
(
    dim_ticker.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mercado_financeiro.gold.dim_ticker")


)

In [0]:
#verificando o data type da coluna data - silver to gold 
df_silver.printSchema()

In [0]:
#convertendo timestamp para data - silver to gold
from pyspark.sql import functions as F
df_gold = (
    df_silver
    .withColumn("Data", F.to_date("Data"))
)

In [0]:
#visualizando após timestamp para data - silver to gold
df_gold.show(5)
df_gold.printSchema()

In [0]:
#salvar na camada gold 

(
   df_gold.write
   .format("delta")
   .mode("overwrite")
   .saveAsTable("mercado_financeiro.gold.ibvespa_acoes")

)

In [0]:
#ler a tabela gold
df_gold_data = spark.table(
    "mercado_financeiro.gold.ibvespa_acoes"
)


In [0]:
#criar a tabela dim_calendario
dim_data = (
    df_gold_data
    .select("Data")
    .distinct()
    .withColumn("ano", F.year("Data"))
    .withColumn("trimestre", F.quarter("Data"))
    .withColumn("mes", F.month("Data"))
    .withColumn("dia", F.dayofmonth("Data"))
    .withColumn("dia_semana_num", F.dayofweek("Data"))
    .withColumn("dia_semana", F.date_format("Data", "EEEE"))
    .withColumn("mês_nome", F.date_format("Data", "MMMM"))

)

In [0]:
#visualizando a dim data
dim_data.show()

In [0]:
#traduzindo01 para pt-br - nome do dia da semana
dim_data = (
    dim_data
    .withColumn(
        "dia_semana",
        F.when(F.col("dia_semana_num") == 2, "segunda-feira")
         .when(F.col("dia_semana_num") == 3, "terça-feira")
         .when(F.col("dia_semana_num") == 4, "quarta-feira")
         .when(F.col("dia_semana_num") == 5, "quinta-feira")
         .when(F.col("dia_semana_num") == 6, "sexta-feira")
    )
)

        

In [0]:
dim_data.show()

In [0]:
#traduzindo02 para pt-br - nome do mes
dim_data = (
    dim_data
    .withColumn(
        "mês_nome",
        F.when(F.col("mes") == 1, "janeiro")
         .when(F.col("mes") == 2, "fevereiro")
         .when(F.col("mes") == 3, "março")
         .when(F.col("mes") == 4, "abril")
         .when(F.col("mes") == 5, "maio")
         .when(F.col("mes") == 6, "junho")
         .when(F.col("mes") == 7, "julho")
         .when(F.col("mes") == 8, "agosto")
         .when(F.col("mes") == 9, "setembro")
         .when(F.col("mes") == 10, "outubro")
         .when(F.col("mes") == 11, "novembro")
         .when(F.col("mes") == 12, "dezembro"))
)

In [0]:
#visualizando a dim_data após a transformação

dim_data.show()

In [0]:
#salvar a dim_data na gold
(
dim_data.write
   .format("delta")
   .mode("overwrite")
   .saveAsTable("mercado_financeiro.gold.dim_data")
   )


In [0]:
#ler a tabela gold

fato = spark.table("mercado_financeiro.gold.ibvespa_acoes")

In [0]:
#criar a tabela fato cotações

from pyspark.sql.functions import col

fato = fato.select(
       col("Data"),
       col("Codigo"),
       col("Abertura"),
       col("Maximo"),
       col("Minimo"),
       col("Fechamento"),
       col("Fechamento_Ajustado"),
       col("Volume")
)


In [0]:
#visualizando após a criação
fato.show(2)

In [0]:
#salvando a tabela fato cotações na camada gold
(

    fato.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mercado_financeiro.gold.fato_cotacoes")


)